In [ ]:
%pip install azure-ai-projects
%pip install azure-identity
%pip install python-dotenv
%pip install agent-framework
%pip install aiohttp

In [29]:
from azure.identity import AzureCliCredential, OnBehalfOfCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, ConnectionType, Connection
from agent_framework.foundry import FoundryAgent
from dotenv import load_dotenv
import os
import json

In [2]:
load_dotenv(override=True)

PROJECT_ENDPOINT=os.getenv('PROJECT_ENDPOINT')

In [27]:
project_client = AIProjectClient(
  endpoint=PROJECT_ENDPOINT,
  credential=AzureCliCredential())

In [ ]:
mcp_con = Connection(
    name="test"
)

In [31]:
connections = project_client.connections.list()

for con in connections:
    print(json.dumps(con.as_dict(),indent=4))

{
    "name": "FlightMCPServer",
    "id": "/subscriptions/6e37307e-394c-478a-8404-4e441b3dfc1d/resourceGroups/rg-multi-agent-demo/providers/Microsoft.CognitiveServices/accounts/cog-penkryleu3m3e/projects/cog-penkryleu3m3e-travel-planner/connections/FlightMCPServer",
    "type": "RemoteTool",
    "target": "https://app-mcp-fligh-server-penkryleu3m3e.azurewebsites.net/mcp",
    "isDefault": true,
    "credentials": {
        "type": "OAuth2"
    },
    "metadata": {
        "type": "custom_MCP"
    }
}


In [ ]:
mcp_con = Connection(
    name="test",
    type="RemoteTool",
    target="https://app-mcp-fligh-server-penkryleu3m3e.azurewebsites.net/mcp"    
)

In [9]:
tool = MCPTool(
    server_label="FlightServerMCP",
    server_url="https://app-mcp-fligh-server-penkryleu3m3e.azurewebsites.net/mcp",
    require_approval="never",
    project_connection_id="FlightMCPServer"
)

In [16]:
created_agent = project_client.agents.create_version(
    agent_name="FlightBookingAgent",
    definition=PromptAgentDefinition(
    model="gpt-5.4-mini",
    instructions="You are a FlightBookingAgent and you use always the MCP Tool called `FlightServerMCP`",
    tools=[tool])
)

In [23]:
client = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=created_agent.name,
    allow_preview=True,
    agent_version=created_agent.version,
    credential=AzureCliCredential()   
)


In [24]:
session = client.create_session()

In [25]:
response = await client.run("Give me all the flight to Paris",session=session)

In [26]:
print(response.text)

Here are all available flights to **Paris**:

1. **FL-001 — Air France AF345**
   - From: Montreal (YUL)
   - To: Paris (CDG)
   - Departure: 2026-05-26 01:45 UTC
   - Arrival: 2026-05-26 14:30 UTC
   - Duration: 6.75 hours
   - Stops: 0
   - Cabin: Economy
   - Price: **CAD 685**
   - Seats: 12

2. **FL-002 — Air Canada AC870**
   - From: Montreal (YUL)
   - To: Paris (CDG)
   - Departure: 2026-05-26 04:15 UTC
   - Arrival: 2026-05-26 17:45 UTC
   - Duration: 7.5 hours
   - Stops: 0
   - Cabin: Economy
   - Price: **CAD 720**
   - Seats: 8

3. **FL-003 — Air Transat TS126**
   - From: Montreal (YUL)
   - To: Paris (CDG)
   - Departure: 2026-05-26 00:30 UTC
   - Arrival: 2026-05-26 14:05 UTC
   - Duration: 7.58 hours
   - Stops: 0
   - Cabin: Economy
   - Price: **CAD 545**
   - Seats: 15

4. **FL-006 — WestJet WS18**
   - From: Toronto (YYZ)
   - To: Paris (CDG)
   - Departure: 2026-05-26 03:30 UTC
   - Arrival: 2026-05-26 17:00 UTC
   - Duration: 7.5 hours
   - Stops: 0
   - Cabin: E